# Agente de revisión de documentos de ingeniería (modelo gratis en Kaggle)
Corre **sin API ni costo**: un modelo abierto (Qwen2.5) dentro de la GPU de Kaggle.

Antes de ejecutar:
1. *Settings → Accelerator*: **GPU T4 x2** (o P100).
2. *Settings → Internet*: **On** (solo para descargar el modelo y los paquetes la primera vez).
3. Sube tus documentos como Dataset (PDF con texto, DOCX, XLSX, CSV, TXT, MD).
4. Sube `doc_review_agent.py`, `local_llm.py` y `checklist_default.md` como Dataset (o súbelos al panel de archivos del notebook).

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes pypdf python-docx openpyxl

In [ ]:
import sys, glob, os
hits = glob.glob("/kaggle/input/**/doc_review_agent.py", recursive=True) + glob.glob("/kaggle/working/doc_review_agent.py")
sys.path.insert(0, os.path.dirname(hits[0]))
from doc_review_agent import ReviewConfig, review_folder
from local_llm import LocalLLM

In [ ]:
# Modelo: 7B en 4 bits cabe en una T4 (~6 GB) y da buena calidad en español.
# Más rápido y liviano: "Qwen/Qwen2.5-3B-Instruct". Más preciso (más lento): "Qwen/Qwen2.5-14B-Instruct" (4 bits).
MODEL = "Qwen/Qwen2.5-7B-Instruct"
llm = LocalLLM(MODEL, load_4bit=True)

In [ ]:
cfg = ReviewConfig(
    extra_instructions="Proyecto: <nombre>. Normas aplicables: <ej. AISC 360-16, NCh433>. Unidades: SI.",
)
IN_DIR = "/kaggle/input/documentos-ingenieria"   # ajusta al nombre de tu dataset
resultado = review_folder(IN_DIR, "/kaggle/working/informe", cfg, llm=llm)

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("/kaggle/working/informe/informe.md", encoding="utf-8").read()))

## Chat con los documentos
Haz preguntas, pide resúmenes o consulta los hallazgos de la revisión. Responde solo con lo que encuentra en los documentos y cita archivo y página.

In [ ]:
from doc_chat import DocChat
chat = DocChat(llm)
chat.add_folder(IN_DIR)
chat.add_review("/kaggle/working/informe/revision.json")   # opcional: permite preguntar por los hallazgos

In [ ]:
print(chat.ask("¿Cuáles son los hallazgos críticos y mayores?"))
print(chat.ask("¿Qué norma se aplica y qué factores de seguridad se usan?"))

In [ ]:
print(chat.summarize())            # todos los documentos; o chat.summarize("archivo.pdf")

In [ ]:
chat.repl()   # chat interactivo: escribe tus preguntas; /resumen, /reset, /salir